# 12 · SkewerJAX and fake_spectra: the results compared

fake_spectra is the external reference code for spectra from simulations. This notebook compares what the two codes
compute on the same simulation skewers: the number density along the skewer and the transmitted flux $F = e^{-\tau}$.

fake_spectra runs natively here, called from the installed package (kernel `skewerjax-fs`, recipe in `docs/installation.md`),
and every difference between the two codes is one option of the SkewerJAX function it belongs to.

The path: one skewer through the core calls with the defaults; the same skewer in fake_spectra; the same calls with every
option at fake_spectra's value, the matched mode; then each option on its own, from both ends, and what it weighs.

In [ ]:
import contextlib, io, os, tempfile
os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ.setdefault("OMP_NUM_THREADS", str(os.cpu_count()))  # fake_spectra's OpenMP threads
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import SWITCH_COLORS, plot_code_comparison, plot_dag, use_backend, use_style
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()        # paper style: Tol's colour-blind-safe "bright" colours and colour maps, minimum font sizes

In [ ]:
plot_dag(('simulation', 'tau', 'fake_spectra',), external=True); plt.show()

## One skewer: the SkewerJAX default

Along $y$ from end to end. A `SkewerBundle` holds the path and its frame,
with the snapshot's $a$ at the middle of the path and $a(x)$ from the cosmology; `select_cells_near_paths` the raw cells
within reach, `gas_to_cells` their gas physics, `cell_pieces` the pieces and `compute_optical_depth` $\tau$. Every default
is written in the calls, since they are the options of the next sections.
The pixels are fake_spectra's: $n_{\rm bins} = \lfloor v_{\max}/1\ \mathrm{km/s}\rfloor$, $dv = v_{\max}/n_{\rm bins}$,
$v_{\max} = (aH)_{\rm snap}L$

In [ ]:
from skewerjax.core import (Cells, LineBundle, SkewerBundle, SnapshotFrame, axis_aligned_skewer_points, cell_pieces, compute_optical_depth,
                            density_per_pixel, path_geometry, scale_factors_along_paths)
from skewerjax.simulations import ArepoHDF5Reader, gas_to_cells, select_cells_near_paths
from skewerjax.simulations.gas_physics import fake_spectra_sf_threshold
from skewerjax.utils.paths import DATA_DIR as DATA      # folder with the snapshots: $SKEWERJAX_DATA (docs/installation.md)

In [ ]:
lines = LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA")
SPECIES = list(lines.species_keys)
LINE_OF = {"HI": 0, "SiIII": 1}

In [ ]:
SIM, SNAP, AXIS, SPACING_KMS = "TNG50-2", 25, 2, 0.25     # along y (axis 2); points every 0.25 km/s
reader = ArepoHDF5Reader(f"{DATA}/{SIM}", SNAP)
H = reader.header
L, AH, a_snap = H.box_cMpc, H.velfac_kms_per_cMpc, H.a     # [cMpc], (aH)_snap [km/s per cMpc]
nbins, dv = H.pixel_grid(1.0)                              # fake_spectra's pixels
v = (np.arange(nbins) + 0.5) * dv                          # [km/s], over [0, v_max)
V_PERIOD = nbins * dv

SNAP_FRAME = SnapshotFrame(a_snap, AH, path_length=np.array([L]))            # the frame of the matched mode

print(f"z = {H.redshift:.3f}, L = {L:.3f} cMpc, {H.n_gas:,} gas cells; {nbins} pixels of {dv:.6f} km/s")

In [ ]:
def skewers_along_y(x, z):
    # (N_sk,) x and z [cMpc]: the points from y = 0 to L and the bundle: a_snap at the middle of the path, a(x) from the cosmology
    points = axis_aligned_skewer_points(np.column_stack([x, np.zeros_like(x), z]), np.full(np.size(x), AXIS), L, SPACING_KMS / AH)
    s = np.asarray(path_geometry(jnp.asarray(points))["s"])
    a_first = scale_factors_along_paths(s, a_snap, H.cosmology(), s_ref=np.full(np.size(x), L / 2))[:, 0]
    bundle = SkewerBundle(
        spatial_points=jnp.asarray(points), a0=a_first, x0=0.0, ICs_defined_at_index=0,
        ICs_at_emission_or_absorption="absorption", cosmologies=H.cosmology()
    )
    return points, bundle

points, bundle = skewers_along_y(np.array([L / 4]), np.array([L / 4]))

print(f"{points.shape[1]:,} points; a from {float(bundle.scale_factors[0, 0, 0]):.5f} to {float(bundle.scale_factors[0, 0, -1]):.5f} (a_snap = {a_snap:.5f})")

In [ ]:
gas = select_cells_near_paths(reader, points)                                # the raw cells within reach of the path

In [ ]:
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="amu",             
    T_constants="codata",     
    x_HI_model="stored_sf",           
    cloudy_density="hydrogen",  
    cloudy_interpolation="linear",   
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    skewer_bundle=bundle,
    deposition="voronoi",       
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=False,         
    voigt="ExoJAX",                   
    detuning="exact",                
    a_eff=None,                     
    hubble_flow="exact",
    aH_ref=None,  
    b_constants="codata",       
    pixel_average=False,               
    tautail=None,                     
    resolution_tolerance=0.25,
    piece_chunk=2048,
)

In [ ]:
T = jnp.exp(-tau)
tau0 = np.asarray(tau)

print(f"<F> = {np.exp(-tau0[0]).mean():.4f} (Lya), {np.exp(-tau0[1]).mean():.6f} (Si III)")
print(f"pixels: {nbins} of {dv:.9f} km/s, v_max {V_PERIOD:.6f}")

In [ ]:
def densities(parts):
    # {species: (N_sk, nbins)} n [physical cm^-3 at the snapshot] on fake_spectra's pixels (below)
    return {sp: density_per_pixel(parts["absorbers"][sp], L, nbins) for sp in SPECIES}

y = (np.arange(nbins) + 0.5) * L / nbins                       # [cMpc], the centres of the position pixels of n
rms = lambda d: float(np.sqrt(np.nanmean(np.square(d))))

def result(tau, n):
    # {species: n, tau, F on the central skewer, with their grids}: one mode, as the figures take it
    tau = np.asarray(tau)
    return {sp: dict(n=n[sp][0], tau=tau[LINE_OF[sp], 0], F=np.exp(-tau[LINE_OF[sp], 0]), v=v, y=y) for sp in SPECIES}

n0 = densities(parts)
DEFAULT = result(tau0, n0)

The number density is the mean physical density in equal **position** pixels of the path, at the snapshot
(`density_per_pixel`: the exact overlaps of the pieces with each pixel, over its physical length), so its axis is the
comoving position $y$. The figures draw it above $\tau$ and $F$, which are against $v$, with the same horizontal extent
through $y = v/(aH)_{\rm snap}$.

## The same skewer in fake_spectra

`Spectra` is called directly, on the same snapshot and sightline and with the same pixel width. Its sightline is an axis
(1, 2, 3 = $x$, $y$, $z$) and a point in code units (ckpc/$h$): printed next to SkewerJAX's in cMpc so they can be
compared. Its options stay at its defaults for TNG: `sf_neutral=True` and the top-hat kernel.
It also offers a partial reconstruction of the Voronoi mesh, `kernel="voronoi"`.
$n$ is `get_density`, `get_col_density` over the physical pixel length, the same quantity as above.

In [ ]:
from fake_spectra.spectra import Spectra
WORK = tempfile.mkdtemp(prefix="fs_work_")

def quiet(f, *args):
    # fake_spectra prints its progress per file of the snapshot, whatever quiet=
    with contextlib.redirect_stdout(io.StringIO()):
        return f(*args)

def native_spectra(x, z, name):
    # Spectra for the sightlines along y at (x, z) [cMpc]: axis and cofm in its units
    cofm = np.column_stack([x, np.zeros_like(x), z]) / H.code_length_cMpc
    return Spectra(SNAP, f"{DATA}/{SIM}", cofm, np.full(np.size(x), AXIS, dtype=np.int32), res=1.0, savefile=f"{name}.hdf5", savedir=WORK, reload_file=True, quiet=True)

In [ ]:
fs = native_spectra(np.array([L / 4]), np.array([L / 4]), "centre")
u = (np.arange(fs.nbins) + 0.5) * fs.dvbin                     # [km/s], fake_spectra's own velocity pixels
FS = {
    "HI": dict(tau=np.asarray(quiet(fs.get_tau, "H", 1, 1215))[0], n=np.asarray(quiet(fs.get_density, "H", 1))[0]),
    "SiIII": dict(tau=np.asarray(quiet(fs.get_tau, "Si", 3, 1206))[0], n=np.asarray(quiet(fs.get_density, "Si", 3))[0])
}
for f in FS.values():
    f.update(F=np.exp(-f["tau"]), v=u, y=y)
    
print(f"{os.environ['OMP_NUM_THREADS']} OpenMP threads; kernel {fs.kernel_int} (0 = top hat), sf_neutral = {fs.gasprop.sf_neutral}")
print(f"sightline, fake_spectra: axis {fs.axis[0]}, cofm {fs.cofm[0]} ckpc/h = {fs.cofm[0] * H.code_length_cMpc} cMpc")
print(f"pixels: {fs.nbins} of {fs.dvbin:.9f} km/s, v_max {fs.vmax:.6f}; max |u - v| = {np.abs(u - v).max():.1e} km/s")

## One skewer: the SkewerJAX matched

The same calls as in the default block, with every option written at fake_spectra's value and a comment on what it changes.
The next section takes these options one by one.

In [ ]:
cells_m = gas_to_cells(
    gas, SPECIES,
    mass_unit="proton",             # n = rho X / (A m_p), not A m_u
    T_constants="fake_spectra",     # its rounded m_p and k_B in T
    x_HI_model="fake_sf",           # sf_neutral=True: its density threshold and its UVB
    cloudy_density="fake_spectra",  # hden = 0.774 rho / m_p, not n_H
    cloudy_interpolation="cubic",   # a spline of log f, not bilinear
)
pieces_m, parts_m = cell_pieces(
    cells_m, lines,
    points=points,
    frame=SNAP_FRAME,                 # one a and u = aH s + v_pec, not the bundle's a(x)
    deposition="tophat_chords",       # spheres of radius V^(1/3) and 7 points per chord, not the Voronoi cells
    chord_weight=1 / 8,               # its SPH weights: 7/8 of the column, in tau only
    a_data=a_snap, return_parts=True,
)
tau_m = compute_optical_depth(
    pieces_m, v, lines,
    velocity_period=V_PERIOD,         # periodic by construction, not a direct integral
    voigt="harris",                   # the first order in the damping parameter, not ExoJAX
    detuning="linear",                # H(a, du / b), not the exact map to frequency
    a_eff=a_snap,                     # one comoving-to-physical conversion, not a_p^-2 along the path
    hubble_flow="linear",
    aH_ref=AH,                        # the line centres on (aH)_snap y, not c ln(a_dagger / a(y))
    b_constants="fake_spectra",       # b^2 = 2 k_B T / (A m_p) with the constants of its C engine
    pixel_average=True,               # cells with b < 2 dv averaged over the pixel, not the profile at its centre
    tautail=1e-7,                     # each cell stops below 1e-7, not its full profile
    resolution_tolerance=None, piece_chunk=2048,
)

print(f"<F> = {np.exp(-np.asarray(tau_m[0])).mean():.4f} (Lya), {np.exp(-np.asarray(tau_m[1])).mean():.6f} (Si III)")

In [ ]:
def tophat_density(cells):
    # n when tau uses the chord points: fake_spectra's column keeps the exact overlaps of the spheres (absorption.cpp L211)
    return densities(cell_pieces(cells, lines, skewer_bundle=bundle, deposition="tophat", a_data=a_snap, return_parts=True)[1])

MATCHED = result(tau_m, tophat_density(cells_m))

In [ ]:
ZOOM = None # the window of every figure below: None (the whole skewer) or e.g. (1500, 2500) [km/s] or (V_PERIOD-1000, V_PERIOD)
for sp in SPECIES:
    fig, axs = plot_code_comparison(
        [dict(mode="fake_spectra", **FS[sp]), dict(mode="default", **DEFAULT[sp]), dict(mode="matched", **MATCHED[sp])],
        residuals=[("default", "fake_spectra"), ("matched", "fake_spectra")], linewidths=(4, 2.5, 1.5), residual_limits=dict(dF=(1e-7, 1e-15)) if sp == "SiIII" else None, aH=AH, species=sp, window=ZOOM,
        title=f"{lines.lines[LINE_OF[sp]].id}: the matched mode reproduces fake_spectra, rms(ΔF) = {rms(MATCHED[sp]['F'] - FS[sp]['F']):.1e}, against {rms(DEFAULT[sp]['F'] - FS[sp]['F']):.1e} for the default")
    plt.show()

## The switches, one figure each

Every difference is one option, and each is tested on its own, from both ends:

- **A** is the default with only this option at fake_spectra's value: what this choice does to SkewerJAX;
- **B** is the matched mode with only this option back at SkewerJAX's default: how far fake_spectra's result moves without it.

Nothing accumulates: every step starts again from the default and from the matched mode above. The steps follow the route of the gas:

1. **Extraction** (`gas_to_cells`, steps 1–5): the gas physics per cell, before any geometry. The constants first, then
   the ionisation that depends on them.
2. **Deposition** (`cell_pieces`, steps 6–8): where that gas sits along the path.
3. **Integration** (`compute_optical_depth`, steps 9–17): first what the profile of one piece is ($b$, Voigt), then the
   spectral axis (period, frame, detuning), and last the two per-cell schemes on that axis: the pixel average, and
   `tautail`, whose walk runs over the pixel averages.

A few options need another one, and take it along in both directions, marked `*` in the cell and the title:

| step | A takes along | B takes along |
|---|---|---|
| 6, `deposition="tophat"` | | `chord_weight`: without spheres there are no chords |
| 7, `deposition="tophat_chords"` | the spheres: the chords are chords of them | `chord_weight`: without chord points there are no weights |
| 8, `chord_weight=1/8` | `deposition="tophat_chords"` | |
| 12, `a_eff` | | the `ExactFrame`: in `SnapshotFrame`, $a = a_{\rm snap}$ everywhere, so `a_eff=None` changes nothing |
| 13, `hubble_flow` | | the `ExactFrame`: in `SnapshotFrame` the flow is in the velocities, so `"exact"` changes nothing |
| 14, `SnapshotFrame` | steps 12 and 13 | |

Each cell has the same calls as above, every option written: first A, then B, with `# <-- switched` on the option and `# <-- taken along` on what it needs. Each figure has four
curves, the default, the matched mode, A and B, and two residuals, A against the default and B against the matched mode. fake_spectra is not drawn: the
matched mode stands for it, since it reproduces it to $\le 10^{-5}$ in $F$ (above). For $n$ and $\tau$ the residual is $2(a-b)/(a+b)$ over the pixels with gas, written
$\Delta n/\bar n$ and $\Delta\tau/\bar\tau$; for $F$ it is $\Delta F$. The residual rows have the same limits in all the figures.

Each curve of $\tau$ and $F$ is on the $v$ of its own mode, which is $c\ln(a_\dagger/a(y))$ until `hubble_flow="linear"` (step 13). The pixels are the same numbers, the features differ
slightly, and this is left uncorrected: it is one of the differences being measured.

In [ ]:
A, B = {}, {}    # {switch: result(...)}: A the default + this switch, B the matched mode - this switch

def step_figure(step, switch, species="HI", star=False):
    # the four curves and two residuals of one step; the matched mode stands for fake_spectra
    a, b, j = A[switch][species], B[switch][species], LINE_OF[species]
    fig, axs = plot_code_comparison(
        [dict(mode="default", **DEFAULT[species]), dict(mode="matched", **MATCHED[species]),
         dict(mode="default+switch", **a), dict(mode="matched-switch", **b)],
        residuals=[("default+switch", "default"), ("matched-switch", "matched")], linewidths=(4, 3.5, 3, 2.5),
        aH=AH, species=species, window=ZOOM,
        title=f"Step {step}, {switch}{' *' if star else ''}, {lines.lines[j].id}: rms(ΔF) = {rms(a['F'] - DEFAULT[species]['F']):.2e} (A − default), {rms(b['F'] - MATCHED[species]['F']):.2e} (B − matched)")
    plt.show()

**Step 1, `T_constants`.** The temperature, $T = (\gamma-1)\,\mu\,m_p u/k_B$: the same formula with other numbers.

- SkewerJAX: CODATA 2018, $m_p = 1.67262192\times10^{-24}$ g, $k_B = 1.380649\times10^{-16}$ erg/K | fake_spectra: $m_p = 1.67262178\times10^{-24}$ g, $k_B = 1.38066\times10^{-16}$ erg/K (`unitsystem.py` L21, L23; `abstractsnapshot.py` L153).
- Only rounding: $T$ lower by $8.1\times10^{-6}$, $b$ by $4\times10^{-6}$; the CLOUDY table is read at that $T$ too. Negligible.

In [ ]:
# step 1: T_constants
# A: the default with T_constants at fake_spectra's value
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="amu",
    T_constants="fake_spectra",      # <-- switched
    x_HI_model="stored_sf",
    cloudy_density="hydrogen",
    cloudy_interpolation="linear",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    skewer_bundle=bundle,
    deposition="voronoi",
    chord_weight=None,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=False,
    voigt="ExoJAX",
    detuning="exact",
    a_eff=None,
    hubble_flow="exact",
    aH_ref=None,
    b_constants="codata",
    pixel_average=False,
    tautail=None,
    resolution_tolerance=0.25,
    piece_chunk=2048,
)
A["T_constants"] = result(tau, densities(parts))
# B: the matched mode with T_constants back at SkewerJAX's value
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="proton",
    T_constants="codata",            # <-- switched
    x_HI_model="fake_sf",
    cloudy_density="fake_spectra",
    cloudy_interpolation="cubic",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    frame=SNAP_FRAME,
    deposition="tophat_chords",
    chord_weight=1 / 8,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=V_PERIOD,
    voigt="harris",
    detuning="linear",
    a_eff=a_snap,
    hubble_flow="linear",
    aH_ref=AH,
    b_constants="fake_spectra",
    pixel_average=True,
    tautail=1e-7,
    resolution_tolerance=None,
    piece_chunk=2048,
)
B["T_constants"] = result(tau, tophat_density(cells))
step_figure(1, "T_constants")

**Step 2, `mass_unit`.** The number density from the mass density.

- SkewerJAX: $n = \rho X/(A\,m_u)$, $m_u = 1.66053907\times10^{-24}$ g | fake_spectra: $n = \rho X/(A\,m_p)$, $m_p = 1.67262178\times10^{-24}$ g, $m_p/m_u = 1.00728$ (`gas_properties.py` L108, `spectra.py` L643; `absorption.h` L4 labels the proton mass "1 a.m.u").
- $A$ is an atomic weight, in units of $m_u$ (H: 1.00794, the same in both), so $m_u$ is the unit it needs. Every density and $\tau$ goes down by 0.72 %, and so does $n_{\rm H}$, which also reads the CLOUDY table (step 4).

In [ ]:
# step 2: mass_unit
# A: the default with mass_unit at fake_spectra's value
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="proton",              # <-- switched
    T_constants="codata",
    x_HI_model="stored_sf",
    cloudy_density="hydrogen",
    cloudy_interpolation="linear",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    skewer_bundle=bundle,
    deposition="voronoi",
    chord_weight=None,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=False,
    voigt="ExoJAX",
    detuning="exact",
    a_eff=None,
    hubble_flow="exact",
    aH_ref=None,
    b_constants="codata",
    pixel_average=False,
    tautail=None,
    resolution_tolerance=0.25,
    piece_chunk=2048,
)
A["mass_unit"] = result(tau, densities(parts))
# B: the matched mode with mass_unit back at SkewerJAX's value
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="amu",                 # <-- switched
    T_constants="fake_spectra",
    x_HI_model="fake_sf",
    cloudy_density="fake_spectra",
    cloudy_interpolation="cubic",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    frame=SNAP_FRAME,
    deposition="tophat_chords",
    chord_weight=1 / 8,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=V_PERIOD,
    voigt="harris",
    detuning="linear",
    a_eff=a_snap,
    hubble_flow="linear",
    aH_ref=AH,
    b_constants="fake_spectra",
    pixel_average=True,
    tautail=1e-7,
    resolution_tolerance=None,
    piece_chunk=2048,
)
B["mass_unit"] = result(tau, tophat_density(cells))
step_figure(2, "mass_unit")

**Step 3, `x_HI_model`.** The neutral fraction of star-forming gas; every other cell keeps the stored fraction in both codes.

- SkewerJAX: where SFR > 0, Rahmati+13 at $10^4$ K with $n_{\rm H}$ and TNG's UVB, $\Gamma_{\rm HI} = 5.48\times10^{-13}\ {\rm s^{-1}}$ at this $z$ (TREECOOL) | fake_spectra: where $\rho/m_p > 0.1199/0.76 = 0.158\ {\rm cm^{-3}}$, the same fit with $\rho/m_p$ in place of $n_{\rm H}$ and the FG09 rate, $\Gamma_{\rm HI} = 5.52\times10^{-13}\ {\rm s^{-1}}$ (`spectra.py` L86, `gas_properties.py` L44–L46, L138–L145).
- Three differences: which cells (the simulation's flag or a density threshold), the density in the fit, and the rate. H I only, and only in such cells, so the effect is exactly 0 on a skewer that crosses none: the cell prints how many it crosses.

In [ ]:
# step 3: x_HI_model
def crossed(parts):
    # the ids of the cells these pieces belong to
    c = parts["segments"].cell
    return np.asarray(parts["cells"].ids)[np.unique(c[c >= 0])]

# A: the default with x_HI_model at fake_spectra's value
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="amu",
    T_constants="codata",
    x_HI_model="fake_sf",            # <-- switched
    cloudy_density="hydrogen",
    cloudy_interpolation="linear",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    skewer_bundle=bundle,
    deposition="voronoi",
    chord_weight=None,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=False,
    voigt="ExoJAX",
    detuning="exact",
    a_eff=None,
    hubble_flow="exact",
    aH_ref=None,
    b_constants="codata",
    pixel_average=False,
    tautail=None,
    resolution_tolerance=0.25,
    piece_chunk=2048,
)
A["x_HI_model"] = result(tau, densities(parts))
ids_voronoi = crossed(parts)
# B: the matched mode with x_HI_model back at SkewerJAX's value
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="proton",
    T_constants="fake_spectra",
    x_HI_model="stored_sf",          # <-- switched
    cloudy_density="fake_spectra",
    cloudy_interpolation="cubic",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    frame=SNAP_FRAME,
    deposition="tophat_chords",
    chord_weight=1 / 8,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=V_PERIOD,
    voigt="harris",
    detuning="linear",
    a_eff=a_snap,
    hubble_flow="linear",
    aH_ref=AH,
    b_constants="fake_spectra",
    pixel_average=True,
    tautail=1e-7,
    resolution_tolerance=None,
    piece_chunk=2048,
)
B["x_HI_model"] = result(tau, tophat_density(cells))
ids_spheres = crossed(parts)
step_figure(3, "x_HI_model")
rho_mp = np.asarray(gas.fields["rho"]) / 1.67262178e-24
sf, dense = np.asarray(gas.fields["sfr"]) > 0, rho_mp > fake_spectra_sf_threshold(H.h) / 0.76
on_path, on_spheres = np.isin(gas.ids, ids_voronoi), np.isin(gas.ids, ids_spheres)
SF_CROSSED = int((on_path & (sf | dense)).sum())   # the root-cause table reads it
print(f"this skewer crosses {on_path.sum():,} Voronoi cells: {(on_path & sf).sum()} with SFR > 0, {(on_path & dense).sum()} above fake_spectra's threshold; "
      f"it touches {on_spheres.sum():,} spheres: {(on_spheres & sf).sum()} and {(on_spheres & dense).sum()}; densest cell crossed rho / m_p = {rho_mp[on_path].max():.3g} cm^-3")
print(f"of the {len(gas.positions):,} cells near it, {sf.sum()} have SFR > 0, {dense.sum()} are above the threshold")

**Step 4, `cloudy_density`.** The density at which the CLOUDY table $f(h_{\rm den}, T)$ is read. Only metals: this figure shows Si III.

- SkewerJAX: $h_{\rm den} = n_{\rm H} = \rho X_{\rm H}/(1.00794\,m_u)$, with each cell's $X_{\rm H}$ | fake_spectra: $h_{\rm den} = 0.774132\,\rho/m_p$, one factor for every cell (`spectra.py` L691–L698, `cloudy_tables/convert_cloudy.py` L183).
- CLOUDY's $h_{\rm den}$ is the hydrogen density. At $X_{\rm H} = 0.76$ fake_spectra's is 1.9 % higher, 0.008 dex against a table step of 0.2 dex; on this skewer that is an rms $\Delta\tau/\tau$ of 8 %. Si III is optically thin by many decades here, so its effect is in $n$ and $\tau$, not in $F$.

In [ ]:
# step 4: cloudy_density
# A: the default with cloudy_density at fake_spectra's value
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="amu",
    T_constants="codata",
    x_HI_model="stored_sf",
    cloudy_density="fake_spectra",   # <-- switched
    cloudy_interpolation="linear",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    skewer_bundle=bundle,
    deposition="voronoi",
    chord_weight=None,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=False,
    voigt="ExoJAX",
    detuning="exact",
    a_eff=None,
    hubble_flow="exact",
    aH_ref=None,
    b_constants="codata",
    pixel_average=False,
    tautail=None,
    resolution_tolerance=0.25,
    piece_chunk=2048,
)
A["cloudy_density"] = result(tau, densities(parts))
# B: the matched mode with cloudy_density back at SkewerJAX's value
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="proton",
    T_constants="fake_spectra",
    x_HI_model="fake_sf",
    cloudy_density="hydrogen",       # <-- switched
    cloudy_interpolation="cubic",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    frame=SNAP_FRAME,
    deposition="tophat_chords",
    chord_weight=1 / 8,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=V_PERIOD,
    voigt="harris",
    detuning="linear",
    a_eff=a_snap,
    hubble_flow="linear",
    aH_ref=AH,
    b_constants="fake_spectra",
    pixel_average=True,
    tautail=1e-7,
    resolution_tolerance=None,
    piece_chunk=2048,
)
B["cloudy_density"] = result(tau, tophat_density(cells))
step_figure(4, "cloudy_density", species="SiIII")

**Step 5, `cloudy_interpolation`.** Between the nodes of the table. Only metals: Si III again.

- SkewerJAX: bilinear in $(\log h_{\rm den}, \log T)$ of $\log_{10} f$ | fake_spectra: a cubic B-spline of $\log_{10} f$ (`map_coordinates`, order 3 by default, `convert_cloudy.py` L199).
- Neither wins everywhere. With every other node of the Si III table left out: where $f > 10^{-4}$, the spline's median error is 10 times smaller ($5\times10^{-4}$ against $5\times10^{-3}$ dex); near the floor $\log_{10} f = -30$, 55 % of the nodes, it rings, up to 5 dex outside its neighbours. This low-density skewer sits next to that floor: its cells differ by an rms of 0.57 dex (0.006 dex weighted by the Si III they hold), an rms $\Delta\tau/\tau$ of 49 %.

In [ ]:
# step 5: cloudy_interpolation
# A: the default with cloudy_interpolation at fake_spectra's value
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="amu",
    T_constants="codata",
    x_HI_model="stored_sf",
    cloudy_density="hydrogen",
    cloudy_interpolation="cubic",    # <-- switched
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    skewer_bundle=bundle,
    deposition="voronoi",
    chord_weight=None,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=False,
    voigt="ExoJAX",
    detuning="exact",
    a_eff=None,
    hubble_flow="exact",
    aH_ref=None,
    b_constants="codata",
    pixel_average=False,
    tautail=None,
    resolution_tolerance=0.25,
    piece_chunk=2048,
)
A["cloudy_interpolation"] = result(tau, densities(parts))
# B: the matched mode with cloudy_interpolation back at SkewerJAX's value
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="proton",
    T_constants="fake_spectra",
    x_HI_model="fake_sf",
    cloudy_density="fake_spectra",
    cloudy_interpolation="linear",   # <-- switched
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    frame=SNAP_FRAME,
    deposition="tophat_chords",
    chord_weight=1 / 8,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=V_PERIOD,
    voigt="harris",
    detuning="linear",
    a_eff=a_snap,
    hubble_flow="linear",
    aH_ref=AH,
    b_constants="fake_spectra",
    pixel_average=True,
    tautail=1e-7,
    resolution_tolerance=None,
    piece_chunk=2048,
)
B["cloudy_interpolation"] = result(tau, tophat_density(cells))
step_figure(5, "cloudy_interpolation", species="SiIII")

**Step 6, `deposition="tophat"`.** Where the gas of each cell sits along the path.

- SkewerJAX: the exact intersections of the path with the Voronoi cells (notebook 07): one density at every point, the cell's $n$ over its real length | fake_spectra: a sphere around each generator of radius $h = V^{1/3}$ and density $3n/4\pi$ (`abstractsnapshot.py` L271, `absorption.cpp` L117), which holds the cell's atoms, $\frac{4\pi}{3}h^3\cdot\frac{3n}{4\pi} = nV$; its chord at impact parameter $d$ is $\ell = 2\sqrt{h^2-d^2}$ (`absorption.cpp` L188). This step takes that chord exactly.
- A mesh replaced by a smoothing kernel. Each sphere is $4\pi/3 = 4.19$ times the volume of its cell, so about 4 overlap at any point (here a mean of 4.18, no gap at all, 2649 spheres against 1228 cells crossed): $n$ is averaged over neighbours, the atoms are kept on average but not the column of a given path. The one switch that moves gas between pixels: rms $\Delta\tau/\tau$ of 11 % here.

In [ ]:
# step 6: deposition='tophat' *
# A: the default with deposition at fake_spectra's spheres
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="amu",
    T_constants="codata",
    x_HI_model="stored_sf",
    cloudy_density="hydrogen",
    cloudy_interpolation="linear",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    skewer_bundle=bundle,
    deposition="tophat",             # <-- switched
    chord_weight=None,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=False,
    voigt="ExoJAX",
    detuning="exact",
    a_eff=None,
    hubble_flow="exact",
    aH_ref=None,
    b_constants="codata",
    pixel_average=False,
    tautail=None,
    resolution_tolerance=0.25,
    piece_chunk=2048,
)
A["deposition='tophat'"] = result(tau, densities(parts))
# B: the matched mode with deposition back at the Voronoi pieces; * chord_weight and the switched-off resolution check go with the chords
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="proton",
    T_constants="fake_spectra",
    x_HI_model="fake_sf",
    cloudy_density="fake_spectra",
    cloudy_interpolation="cubic",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    frame=SNAP_FRAME,
    deposition="voronoi",            # <-- switched
    chord_weight=None,               # <-- taken along
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=V_PERIOD,
    voigt="harris",
    detuning="linear",
    a_eff=a_snap,
    hubble_flow="linear",
    aH_ref=AH,
    b_constants="fake_spectra",
    pixel_average=True,
    tautail=1e-7,
    resolution_tolerance=0.25,       # <-- taken along
    piece_chunk=2048,
)
B["deposition='tophat'"] = result(tau, densities(parts))
step_figure(6, "deposition='tophat'", star=True)

**Step 7, `deposition="tophat_chords"`.** The same spheres, with the integral across each chord in $\tau$ replaced by a quadrature.

- SkewerJAX's exact overlap (step 6): $\int_{-z_r}^{z_r}\phi(v - aH z)\,dz$, $z_r = \sqrt{h^2-d^2} = \ell/2$ | fake_spectra: 7 points $z_k = z_r(k/4-1)$, $k = 1,\dots,7$ (`singleabs.h` L100–L118); here each of weight 1/7, so the column stays.
- The points are $z_r/4$ apart, 6 % of $b$ (median here), so the discreteness is small; but they reach only $\pm\frac{3}{4}z_r$, with a variance $z_r^2/4$ instead of $z_r^2/3$, a slightly narrower absorption. $n$ is that of step 6: fake_spectra's column keeps the exact overlaps (`absorption.cpp` L211). The own effect, net of the spheres, is in the footnote of the table.

In [ ]:
# step 7: deposition='tophat_chords' *
# A: the default with deposition at the chord points; * the spheres come with the chords, and the resolution check goes
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="amu",
    T_constants="codata",
    x_HI_model="stored_sf",
    cloudy_density="hydrogen",
    cloudy_interpolation="linear",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    skewer_bundle=bundle,
    deposition="tophat_chords",      # <-- switched
    chord_weight=None,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=False,
    voigt="ExoJAX",
    detuning="exact",
    a_eff=None,
    hubble_flow="exact",
    aH_ref=None,
    b_constants="codata",
    pixel_average=False,
    tautail=None,
    resolution_tolerance=None,       # <-- taken along
    piece_chunk=2048,
)
A["deposition='tophat_chords'"] = result(tau, tophat_density(cells))
# B: the matched mode with deposition at the exact overlaps of the spheres; * chord_weight and the switched-off resolution check go with the chords
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="proton",
    T_constants="fake_spectra",
    x_HI_model="fake_sf",
    cloudy_density="fake_spectra",
    cloudy_interpolation="cubic",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    frame=SNAP_FRAME,
    deposition="tophat",             # <-- switched
    chord_weight=None,               # <-- taken along
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=V_PERIOD,
    voigt="harris",
    detuning="linear",
    a_eff=a_snap,
    hubble_flow="linear",
    aH_ref=AH,
    b_constants="fake_spectra",
    pixel_average=True,
    tautail=1e-7,
    resolution_tolerance=0.25,       # <-- taken along
    piece_chunk=2048,
)
B["deposition='tophat_chords'"] = result(tau, densities(parts))
step_figure(7, "deposition='tophat_chords'", star=True)

**Step 8, `chord_weight=1/8`.** The weight of each chord point.

- SkewerJAX's chords (step 7): 1/7, so $\sum_k w_k\ell = \ell$ | fake_spectra: $\Delta v = 2v_{\rm high}/8$ per point (`singleabs.h` L100, L116), so $\sum_k w_k\ell = \frac{7}{8}\ell$.
- The weights are meant for an SPH kernel, which vanishes at the two dropped ends (`singleabs.h` L97–L99); a top hat does not, so 1/8 of every chord is lost. In $\tau$ only, not in `colden` (`absorption.cpp` L211): $n$ does not move, and $\tau$ is multiplied by exactly 7/8 (here the mean $\tau$ of A(8) over A(7) is 0.875), the largest own effect on the flux.

In [ ]:
# step 8: chord_weight=1/8 *
# A: the default with chord_weight at fake_spectra's 1/8; * it needs the chords of deposition='tophat_chords' (step 7)
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="amu",
    T_constants="codata",
    x_HI_model="stored_sf",
    cloudy_density="hydrogen",
    cloudy_interpolation="linear",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    skewer_bundle=bundle,
    deposition="tophat_chords",      # <-- taken along
    chord_weight=1 / 8,              # <-- switched
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=False,
    voigt="ExoJAX",
    detuning="exact",
    a_eff=None,
    hubble_flow="exact",
    aH_ref=None,
    b_constants="codata",
    pixel_average=False,
    tautail=None,
    resolution_tolerance=None,       # <-- taken along
    piece_chunk=2048,
)
A["chord_weight=1/8"] = result(tau, tophat_density(cells))
# B: the matched mode with chord_weight back at 1/7, the whole column
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="proton",
    T_constants="fake_spectra",
    x_HI_model="fake_sf",
    cloudy_density="fake_spectra",
    cloudy_interpolation="cubic",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    frame=SNAP_FRAME,
    deposition="tophat_chords",
    chord_weight=None,               # <-- switched
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=V_PERIOD,
    voigt="harris",
    detuning="linear",
    a_eff=a_snap,
    hubble_flow="linear",
    aH_ref=AH,
    b_constants="fake_spectra",
    pixel_average=True,
    tautail=1e-7,
    resolution_tolerance=None,
    piece_chunk=2048,
)
B["chord_weight=1/8"] = result(tau, tophat_density(cells))
step_figure(8, "chord_weight=1/8", star=True)

**Step 9, `b_constants`.** The thermal width, $b^2 = 2k_BT/(A\,m)$, inside the profile only.

- SkewerJAX: CODATA, $k_B/m_u$ | fake_spectra: its C engine's, $k_B = 1.3806504\times10^{-16}$ erg/K (not the $1.38066\times10^{-16}$ of its $T$, step 1) and $m_p$ (`absorption.cpp` L24, L158; `absorption.h` L4).
- The same $m_p$ against $m_u$ as step 2, here in $b$: $b^2$ lower by 0.72 %, $b$ by 0.36 %. The lines are narrower and taller at the same column.

In [ ]:
# step 9: b_constants
# A: the default with b_constants at fake_spectra's value
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="amu",
    T_constants="codata",
    x_HI_model="stored_sf",
    cloudy_density="hydrogen",
    cloudy_interpolation="linear",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    skewer_bundle=bundle,
    deposition="voronoi",
    chord_weight=None,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=False,
    voigt="ExoJAX",
    detuning="exact",
    a_eff=None,
    hubble_flow="exact",
    aH_ref=None,
    b_constants="fake_spectra",      # <-- switched
    pixel_average=False,
    tautail=None,
    resolution_tolerance=0.25,
    piece_chunk=2048,
)
A["b_constants"] = result(tau, densities(parts))
# B: the matched mode with b_constants back at SkewerJAX's value
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="proton",
    T_constants="fake_spectra",
    x_HI_model="fake_sf",
    cloudy_density="fake_spectra",
    cloudy_interpolation="cubic",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    frame=SNAP_FRAME,
    deposition="tophat_chords",
    chord_weight=1 / 8,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=V_PERIOD,
    voigt="harris",
    detuning="linear",
    a_eff=a_snap,
    hubble_flow="linear",
    aH_ref=AH,
    b_constants="codata",            # <-- switched
    pixel_average=True,
    tautail=1e-7,
    resolution_tolerance=None,
    piece_chunk=2048,
)
B["b_constants"] = result(tau, tophat_density(cells))
step_figure(9, "b_constants")

**Step 10, `voigt="harris"`.** The profile.

- SkewerJAX: ExoJAX's Humlicek-type Voigt, within $10^{-6}$ of `wofz` (relative) | fake_spectra: $H \simeq e^{-u^2} + aH_1(u)$ with $H_1$ tabulated for $a \leq 10^{-3}$, the exact function above (`singleabs.h` L57–L66, `fastvoigt.h` L34, L62–L80); SkewerJAX's option falls back on ExoJAX there.
- Against `wofz` the expansion is within $6\times10^{-6}$ for $a \leq 5\times10^{-4}$ and $1.3\times10^{-5}$ at $a = 10^{-3}$ (relative); it neglects $a^2(1-2u^2)e^{-u^2}$. Here every H I piece is below $10^{-3}$ ($a$ from $1.7\times10^{-4}$ to $7.9\times10^{-4}$) and every Si III piece above ($3.5\times10^{-3}$ to $1.6\times10^{-2}$), hence $\Delta F = 0$ for Si III. A matter of speed, not of the result.

In [ ]:
# step 10: voigt
# A: the default with voigt at fake_spectra's expansion
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="amu",
    T_constants="codata",
    x_HI_model="stored_sf",
    cloudy_density="hydrogen",
    cloudy_interpolation="linear",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    skewer_bundle=bundle,
    deposition="voronoi",
    chord_weight=None,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=False,
    voigt="harris",                  # <-- switched
    detuning="exact",
    a_eff=None,
    hubble_flow="exact",
    aH_ref=None,
    b_constants="codata",
    pixel_average=False,
    tautail=None,
    resolution_tolerance=0.25,
    piece_chunk=2048,
)
A["voigt"] = result(tau, densities(parts))
# B: the matched mode with voigt back at ExoJAX
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="proton",
    T_constants="fake_spectra",
    x_HI_model="fake_sf",
    cloudy_density="fake_spectra",
    cloudy_interpolation="cubic",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    frame=SNAP_FRAME,
    deposition="tophat_chords",
    chord_weight=1 / 8,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=V_PERIOD,
    voigt="ExoJAX",                  # <-- switched
    detuning="linear",
    a_eff=a_snap,
    hubble_flow="linear",
    aH_ref=AH,
    b_constants="fake_spectra",
    pixel_average=True,
    tautail=1e-7,
    resolution_tolerance=None,
    piece_chunk=2048,
)
B["voigt"] = result(tau, tophat_density(cells))
step_figure(10, "voigt")

**Step 11, `velocity_period`.** What the spectral axis is.

- SkewerJAX: the direct integral along the path, $\Delta v = v - v_c$ | fake_spectra: the indices wrap modulo the pixels and each cell walks at most half a box each way, the nearest image $\Delta v \to [(\Delta v + P/2) \bmod P] - P/2$, $P = (aH)_{\rm snap}L$ (`absorption.cpp` L253–L281).
- A choice of what is computed, not an approximation: one period of a periodic box against a finite path. The wings and the features that leave one end re-enter at the other, so the effect sits near the two ends of the skewer.

In [ ]:
# step 11: velocity_period
# A: the default with velocity_period at fake_spectra's period
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="amu",
    T_constants="codata",
    x_HI_model="stored_sf",
    cloudy_density="hydrogen",
    cloudy_interpolation="linear",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    skewer_bundle=bundle,
    deposition="voronoi",
    chord_weight=None,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=V_PERIOD,        # <-- switched
    voigt="ExoJAX",
    detuning="exact",
    a_eff=None,
    hubble_flow="exact",
    aH_ref=None,
    b_constants="codata",
    pixel_average=False,
    tautail=None,
    resolution_tolerance=0.25,
    piece_chunk=2048,
)
A["velocity_period"] = result(tau, densities(parts))
# B: the matched mode with velocity_period back at no wrap
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="proton",
    T_constants="fake_spectra",
    x_HI_model="fake_sf",
    cloudy_density="fake_spectra",
    cloudy_interpolation="cubic",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    frame=SNAP_FRAME,
    deposition="tophat_chords",
    chord_weight=1 / 8,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=False,           # <-- switched
    voigt="harris",
    detuning="linear",
    a_eff=a_snap,
    hubble_flow="linear",
    aH_ref=AH,
    b_constants="fake_spectra",
    pixel_average=True,
    tautail=1e-7,
    resolution_tolerance=None,
    piece_chunk=2048,
)
B["velocity_period"] = result(tau, tophat_density(cells))
step_figure(11, "velocity_period")

**Step 12, `a_eff`.** The comoving-to-physical factor of the density.

- SkewerJAX: $\tau \propto n_{\rm com}\,a_p^{-2}\,\ell_p$, the $a$ of each piece | fake_spectra: $a_{\rm snap}^{-2}$ for every piece (`spectra.py` L221, L615).
- The amplitude only: the line centres and the widths stay. Here $a$ goes from 0.2511 to 0.2478 across the box, so $(a_{\rm snap}/a)^2$ goes from 0.987 to 1.013: the default's $\tau$ is 1.3 % below fake_spectra's at the first end, 1.3 % above at the last, and equal in the middle.

In [ ]:
# step 12: a_eff *
# A: the default with a_eff at the snapshot's a
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="amu",
    T_constants="codata",
    x_HI_model="stored_sf",
    cloudy_density="hydrogen",
    cloudy_interpolation="linear",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    skewer_bundle=bundle,
    deposition="voronoi",
    chord_weight=None,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=False,
    voigt="ExoJAX",
    detuning="exact",
    a_eff=a_snap,                    # <-- switched
    hubble_flow="exact",
    aH_ref=None,
    b_constants="codata",
    pixel_average=False,
    tautail=None,
    resolution_tolerance=0.25,
    piece_chunk=2048,
)
A["a_eff"] = result(tau, densities(parts))
# B: the matched mode with a_eff back at None; * SnapshotFrame has a = a_snap everywhere, so a_eff=None alone changes nothing: the ExactFrame comes back too
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="proton",
    T_constants="fake_spectra",
    x_HI_model="fake_sf",
    cloudy_density="fake_spectra",
    cloudy_interpolation="cubic",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    skewer_bundle=bundle,            # <-- taken along
    deposition="tophat_chords",
    chord_weight=1 / 8,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=V_PERIOD,
    voigt="harris",
    detuning="linear",
    a_eff=None,                      # <-- switched
    hubble_flow="linear",
    aH_ref=AH,
    b_constants="fake_spectra",
    pixel_average=True,
    tautail=1e-7,
    resolution_tolerance=None,
    piece_chunk=2048,
)
B["a_eff"] = result(tau, tophat_density(cells))
step_figure(12, "a_eff", star=True)

**Step 13, `hubble_flow="linear"`.** The Hubble part of the line centres.

- SkewerJAX: $V_H(y) = c\ln(a_\dagger/a(y)) = \int_0^y aH\,dy'$ | fake_spectra: $V_H(y) = (aH)_{\rm snap}\,y$ (`absorption.cpp` L237, `spectra.py` L225–L228).
- Positions, where step 12 is amplitudes: the same gas absorbs at another velocity, and the density factor stays. Here the exact flow is up to 2.9 km/s below the linear one in the middle of the box and meets it at both ends, the largest shift of the features. The spacing of the centres changes with them, by $(aH)/(aH)_{\rm snap} = 1 \pm 0.003$, so a uniform gas also changes in $\tau$ per pixel by that much.

In [ ]:
# step 13: hubble_flow *
# A: the default with hubble_flow at fake_spectra's linear flow
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="amu",
    T_constants="codata",
    x_HI_model="stored_sf",
    cloudy_density="hydrogen",
    cloudy_interpolation="linear",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    skewer_bundle=bundle,
    deposition="voronoi",
    chord_weight=None,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=False,
    voigt="ExoJAX",
    detuning="exact",
    a_eff=None,
    hubble_flow="linear",            # <-- switched
    aH_ref=AH,                       # <-- switched
    b_constants="codata",
    pixel_average=False,
    tautail=None,
    resolution_tolerance=0.25,
    piece_chunk=2048,
)
A["hubble_flow"] = result(tau, densities(parts))
# B: the matched mode with hubble_flow back at the exact flow; * SnapshotFrame has its flow in the velocities, so 'exact' alone changes nothing: the ExactFrame comes back too
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="proton",
    T_constants="fake_spectra",
    x_HI_model="fake_sf",
    cloudy_density="fake_spectra",
    cloudy_interpolation="cubic",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    skewer_bundle=bundle,            # <-- taken along
    deposition="tophat_chords",
    chord_weight=1 / 8,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=V_PERIOD,
    voigt="harris",
    detuning="linear",
    a_eff=a_snap,
    hubble_flow="exact",             # <-- switched
    aH_ref=None,                     # <-- switched
    b_constants="fake_spectra",
    pixel_average=True,
    tautail=1e-7,
    resolution_tolerance=None,
    piece_chunk=2048,
)
B["hubble_flow"] = result(tau, tophat_density(cells))
step_figure(13, "hubble_flow", star=True)

**Step 14, `SnapshotFrame`.** What is left of the frame beyond steps 12 and 13.

- SkewerJAX's `ExactFrame`: $v_c = V_H + c\,\mathrm{artanh}(v_{\rm pec}/c)$, the relativistic Doppler shift | fake_spectra: $v_c = (aH)_{\rm snap}\,y + v_{\rm pec}$, a plain sum (`absorption.cpp` L237).
- The frame sets $a = a_{\rm snap}$ everywhere, which is why steps 12 and 13 are no-ops inside it, and adds only this: $c\,\mathrm{artanh}(v/c) - v \simeq (v/c)^2 v/3$, at most $3\times10^{-5}$ km/s here ($|v_{\rm pec}| \leq 205$ km/s). The large A of this figure is steps 12 and 13, taken along; the own effect is B against the matched mode in the title, $\sim10^{-7}$.

In [ ]:
# step 14: frame='snapshot' *
# A: the default with fake_spectra's SnapshotFrame; * it needs a_eff and hubble_flow (steps 12, 13)
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="amu",
    T_constants="codata",
    x_HI_model="stored_sf",
    cloudy_density="hydrogen",
    cloudy_interpolation="linear",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    frame=SNAP_FRAME,                # <-- switched
    deposition="voronoi",
    chord_weight=None,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=False,
    voigt="ExoJAX",
    detuning="exact",
    a_eff=a_snap,                    # <-- taken along
    hubble_flow="linear",            # <-- taken along
    aH_ref=AH,                       # <-- taken along
    b_constants="codata",
    pixel_average=False,
    tautail=None,
    resolution_tolerance=0.25,
    piece_chunk=2048,
)
A["frame='snapshot'"] = result(tau, densities(parts))
# B: the matched mode with the ExactFrame back; a_eff and hubble_flow stay at fake_spectra's, so only the composition of v_pec changes
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="proton",
    T_constants="fake_spectra",
    x_HI_model="fake_sf",
    cloudy_density="fake_spectra",
    cloudy_interpolation="cubic",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    skewer_bundle=bundle,            # <-- switched
    deposition="tophat_chords",
    chord_weight=1 / 8,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=V_PERIOD,
    voigt="harris",
    detuning="linear",
    a_eff=a_snap,
    hubble_flow="linear",
    aH_ref=AH,
    b_constants="fake_spectra",
    pixel_average=True,
    tautail=1e-7,
    resolution_tolerance=None,
    piece_chunk=2048,
)
B["frame='snapshot'"] = result(tau, tophat_density(cells))
step_figure(14, "frame='snapshot'", star=True)

**Step 15, `detuning="linear"`.** The frequency inside the profile.

- SkewerJAX: $\nu = (c/\lambda)\,\mathrm{expm1}(-\Delta v/c)$ | fake_spectra: its first order, $\nu = -(c/\lambda)\,\Delta v/c$, that is $H(a, \Delta v/b)$ (`singleabs.h` L182).
- `hubble_flow` (step 13) places the line centre $v_c$; `detuning` maps the offset $\Delta v = v - v_c$ to a frequency, whatever the centre. The relative difference in $\nu$ is $\Delta v/2c$, of order $b/c$ in the core: $2\times10^{-5}$ at $\Delta v = b$, which makes the profile very slightly asymmetric.

In [ ]:
# step 15: detuning
# A: the default with detuning at fake_spectra's first order
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="amu",
    T_constants="codata",
    x_HI_model="stored_sf",
    cloudy_density="hydrogen",
    cloudy_interpolation="linear",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    skewer_bundle=bundle,
    deposition="voronoi",
    chord_weight=None,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=False,
    voigt="ExoJAX",
    detuning="linear",               # <-- switched
    a_eff=None,
    hubble_flow="exact",
    aH_ref=None,
    b_constants="codata",
    pixel_average=False,
    tautail=None,
    resolution_tolerance=0.25,
    piece_chunk=2048,
)
A["detuning"] = result(tau, densities(parts))
# B: the matched mode with detuning back at the exact map
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="proton",
    T_constants="fake_spectra",
    x_HI_model="fake_sf",
    cloudy_density="fake_spectra",
    cloudy_interpolation="cubic",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    frame=SNAP_FRAME,
    deposition="tophat_chords",
    chord_weight=1 / 8,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=V_PERIOD,
    voigt="harris",
    detuning="exact",                # <-- switched
    a_eff=a_snap,
    hubble_flow="linear",
    aH_ref=AH,
    b_constants="fake_spectra",
    pixel_average=True,
    tautail=1e-7,
    resolution_tolerance=None,
    piece_chunk=2048,
)
B["detuning"] = result(tau, tophat_density(cells))
step_figure(15, "detuning")

**Step 16, `pixel_average`.** What one pixel reports.

- SkewerJAX: the sum of the profiles at the pixel centre | fake_spectra: for a cell with $b < 2\,dv$, that cell's $\tau$ averaged over the pixel on a trapezoid of $2\lceil dv/b\rceil + 1$ points; for $b \geq 2\,dv$, the centre (`singleabs.h` L133–L155).
- A definition of the reported quantity, not an accuracy: a spectrograph integrates the flux $e^{-\tau}$ through its resolution, and neither value is that, since one samples $\tau$ and the other averages $\tau$, and only for narrow cells. Both need the same step for data. On 1 km/s pixels the size is nil: no H I cell qualifies here ($b \geq 7.6$ km/s), and the Si III pieces that do are 45 % of the pieces but $2\times10^{-5}$ of the column.

In [ ]:
# step 16: pixel_average
# A: the default with pixel_average at fake_spectra's value
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="amu",
    T_constants="codata",
    x_HI_model="stored_sf",
    cloudy_density="hydrogen",
    cloudy_interpolation="linear",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    skewer_bundle=bundle,
    deposition="voronoi",
    chord_weight=None,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=False,
    voigt="ExoJAX",
    detuning="exact",
    a_eff=None,
    hubble_flow="exact",
    aH_ref=None,
    b_constants="codata",
    pixel_average=True,              # <-- switched
    tautail=None,
    resolution_tolerance=0.25,
    piece_chunk=2048,
)
A["pixel_average"] = result(tau, densities(parts))
# B: the matched mode with pixel_average back at the pixel centres
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="proton",
    T_constants="fake_spectra",
    x_HI_model="fake_sf",
    cloudy_density="fake_spectra",
    cloudy_interpolation="cubic",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    frame=SNAP_FRAME,
    deposition="tophat_chords",
    chord_weight=1 / 8,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=V_PERIOD,
    voigt="harris",
    detuning="linear",
    a_eff=a_snap,
    hubble_flow="linear",
    aH_ref=AH,
    b_constants="fake_spectra",
    pixel_average=False,             # <-- switched
    tautail=1e-7,
    resolution_tolerance=None,
    piece_chunk=2048,
)
B["pixel_average"] = result(tau, tophat_density(cells))
step_figure(16, "pixel_average")

**Step 17, `tautail`.** How far each cell's profile runs.

- SkewerJAX: every piece on every pixel | fake_spectra: walking out from the pixel of the cell's line centre, both ways, a cell adds pixels up to and including the first one that receives less than $10^{-7}$ from it (`absorption.cpp` L253–L281, `spectra.py` L145).
- A cut per cell, not per pixel, so the wings that add up across cells are lost: the floor of H I's $\tau$ goes from $8.2\times10^{-5}$ to $7.6\times10^{-5}$ here. A cell whose peak is below $10^{-7}$ stops at once: that is all of Si III on this skewer (maximum $\tau = 1.1\times10^{-8}$), which becomes a comb, 52 % of its pixels exactly 0. This cut is what saves time in fake_spectra; SkewerJAX's option saves none, since it evaluates every pixel and then cuts.

In [ ]:
# step 17: tautail
# A: the default with tautail at fake_spectra's threshold
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="amu",
    T_constants="codata",
    x_HI_model="stored_sf",
    cloudy_density="hydrogen",
    cloudy_interpolation="linear",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    skewer_bundle=bundle,
    deposition="voronoi",
    chord_weight=None,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=False,
    voigt="ExoJAX",
    detuning="exact",
    a_eff=None,
    hubble_flow="exact",
    aH_ref=None,
    b_constants="codata",
    pixel_average=False,
    tautail=1e-7,                    # <-- switched
    resolution_tolerance=0.25,
    piece_chunk=2048,
)
A["tautail"] = result(tau, densities(parts))
# B: the matched mode with tautail back at the full profiles
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="proton",
    T_constants="fake_spectra",
    x_HI_model="fake_sf",
    cloudy_density="fake_spectra",
    cloudy_interpolation="cubic",
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    frame=SNAP_FRAME,
    deposition="tophat_chords",
    chord_weight=1 / 8,
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=V_PERIOD,
    voigt="harris",
    detuning="linear",
    a_eff=a_snap,
    hubble_flow="linear",
    aH_ref=AH,
    b_constants="fake_spectra",
    pixel_average=True,
    tautail=None,                    # <-- switched
    resolution_tolerance=None,
    piece_chunk=2048,
)
B["tautail"] = result(tau, tophat_density(cells))
step_figure(17, "tautail")

## Root causes: accuracy, impact and cost of each choice

Each row is an option of a SkewerJAX function; the files and lines of fake_spectra 2.3.0 (`.project_material/fake_spectra`) are in the steps above.
The numbers are what A, the default with that one switch, changes on this skewer: $\mathrm{rms}(\Delta F)$ of each line.
The rank orders the switches by the larger $\mathrm{rms}(\Delta F)$ of the two lines. Three grades:

- **Accuracy** of SkewerJAX's default: better, meh (equivalent in practice), worse, or choice (it depends on the goal).
- **Impact**, from $\mathrm{rms}(\Delta F)$ of $\mathrm{Ly}\alpha$ on this skewer: high above $10^{-3}$, moderate from $10^{-5}$ to $10^{-3}$, low below $10^{-5}$; for steps 7, 8 and 14, the switch's own effect, net of what it takes along.
- **Expected cost** of SkewerJAX's choice against fake_spectra's: low when only a constant or an analytic formula changes, high when clearly more expensive, ? when it needs timing, n/a for a pure choice.

In [ ]:
from IPython.display import Markdown, display

ROOT = [  # (switch, the option at fake_spectra's value, what fake_spectra does, SkewerJAX's choice and its reason), by step
    ('T_constants', '`gas_to_cells(T_constants="fake_spectra")`', 'rounded $m_p$, $k_B$ in $T$', 'CODATA 2018'),
    ('mass_unit', '`gas_to_cells(mass_unit="proton")`', '$n = \\rho X/(A m_p)$', '$A$ is an atomic weight in units of $m_u$; $m_p/m_u = 1.00728$'),
    ('x_HI_model', '`gas_to_cells(x_HI_model="fake_sf")`', 'a density threshold in $\\rho/m_p$; the fit at $\\rho/m_p$ with the FG09 rate', "TNG's own SFR > 0 flag, $n_{\\rm H}$ and its UVB (TREECOOL): the cold phase only where the simulation formed stars"),
    ('cloudy_density', '`gas_to_cells(cloudy_density="fake_spectra")`', '$h_{\\rm den} = 0.774\\,\\rho/m_p$', "$h_{\\rm den} = n_{\\rm H}$, CLOUDY's own coordinate"),
    ('cloudy_interpolation', '`gas_to_cells(cloudy_interpolation="cubic")`', 'a cubic spline of $\\log f$', 'linear in $\\log f$ stays between its neighbours; the spline is closer where the table is smooth, and rings near its $-30$ floor'),
    ("deposition='tophat'", '`cell_pieces(deposition="tophat")`', 'each cell as a sphere of radius $V^{1/3}$, density $3n/4\\pi$', 'Arepo solves the hydrodynamics on the Voronoi cells: the nearest generator is the simulated fluid, piecewise constant and single valued; about 4 spheres overlap at each point'),
    ("deposition='tophat_chords'", '`cell_pieces(deposition="tophat_chords")`', '7 points per chord', 'the pieces are exact intersections: no quadrature of the density'),
    ('chord_weight=1/8', '`cell_pieces(chord_weight=1/8)`', 'SPH weights on a top hat: 7/8 of the column in $\\tau$ (not in `colden`)', 'the column of a cell is a property of the gas: the exact lengths conserve it for any sampling'),
    ('b_constants', '`compute_optical_depth(b_constants="fake_spectra")`', '$b^2 = 2k_BT/(A m_p)$', 'CODATA and $m_u$'),
    ('voigt', '`compute_optical_depth(voigt="harris")`', '$H = e^{-u^2} + aH_1(u)$ for $a \\leq 10^{-3}$', 'ExoJAX to $\\sim10^{-6}$ against `wofz`; the expansion is as good where $a^2$ is small, so this is a matter of speed'),
    ('velocity_period', '`compute_optical_depth(velocity_period=...)`', 'periodic by construction', 'a path is not periodic in velocity by nature: the direct integral, opt-in period'),
    ('a_eff', '`compute_optical_depth(a_eff=a_snap)`', 'one comoving-to-physical conversion', '$a(x)$ along the path: the density of each piece at its own epoch'),
    ('hubble_flow', '`compute_optical_depth(hubble_flow="linear", aH_ref=...)`', '$u = (aH)_{\\rm snap}s$', '$c\\ln(a_\\dagger/a(y))$ from the cosmology: the expansion inside the box shifts the line centres'),
    ("frame='snapshot'", '`SnapshotFrame` in `cell_pieces`', '$v_{\\rm pec}$ added to $u$', 'the relativistic Doppler shift, $c\\,\\mathrm{artanh}(v/c)$; the difference is $(v/c)^2 v/3$'),
    ('detuning', '`compute_optical_depth(detuning="linear")`', '$H(a, \\Delta u/b)$', 'the exact map $\\nu = (c/\\lambda)\\,\\mathrm{expm1}(-\\Delta v/c)$'),
    ('pixel_average', '`compute_optical_depth(pixel_average=True)`', "a cell's profile averaged over the pixel when $b < 2\\,dv$", 'not an accuracy: SkewerJAX gives $\\tau$ at the pixel centres, a convention; a spectrograph integrates $F$, so both need that step'),
    ('tautail', '`compute_optical_depth(tautail=1e-7)`', "stops each cell's profile below $10^{-7}$", 'full Voigt profiles: the wings of a strong absorber and of many weak cells add up'),
]

GRADES = {  # switch: (accuracy of SkewerJAX's default, its expected cost against fake_spectra's); the reasons are in the steps and below
    'T_constants': ('better', 'low'),
    'mass_unit': ('better', 'low'),
    'x_HI_model': ('better', 'low'),
    'cloudy_density': ('better', 'low'),
    'cloudy_interpolation': ('meh', 'low'),
    "deposition='tophat'": ('better', '?'),
    "deposition='tophat_chords'": ('better', '?'),
    'chord_weight=1/8': ('better', 'low'),
    'b_constants': ('better', 'low'),
    'voigt': ('meh', 'high'),
    'velocity_period': ('choice', 'n/a (choice)'),
    'a_eff': ('choice', 'low'),
    'hubble_flow': ('choice', 'low'),
    "frame='snapshot'": ('better', 'low'),
    'detuning': ('better', 'low'),
    'pixel_average': ('choice', 'low'),
    'tautail': ('better', '?'),
}
STARRED = {"deposition='tophat'", "deposition='tophat_chords'", "chord_weight=1/8", "a_eff", "hubble_flow", "frame='snapshot'"}
num = lambda x: "0" if x == 0 else f"{x:.1e}"
dF = {s: {sp: rms(A[s][sp]["F"] - DEFAULT[sp]["F"]) for sp in SPECIES} for s in A}
size = {s: max(dF[s].values()) for s in A}
rank = {s: 1 + sum(size[o] > size[s] for o in A) for s in A}
own = {"deposition='tophat_chords'": ("A(7) − A(6)", A["deposition='tophat_chords'"], A["deposition='tophat'"]),
       "chord_weight=1/8": ("A(8) − A(7)", A["chord_weight=1/8"], A["deposition='tophat_chords'"]),
       "frame='snapshot'": ("B(14) − matched", B["frame='snapshot'"], MATCHED)}
METALS_ONLY = ("cloudy_density", "cloudy_interpolation")
grade = lambda x: "high" if x > 1e-3 else "moderate" if x >= 1e-5 else "low"

def impact(s):
    # from rms(dF) of Lya: the switch's own effect in steps 7, 8, 14; steps 4, 5 add Si III, which they act on
    if s == "x_HI_model" and SF_CROSSED == 0:
        return "n/a (no star-forming cells crossed)"
    if s in own:
        x = rms(own[s][1]["HI"]["F"] - own[s][2]["HI"]["F"])
        return f"{grade(x)} (own: {num(x)})"
    if s in METALS_ONLY:
        ta, t0 = A[s]["SiIII"]["tau"], DEFAULT["SiIII"]["tau"]
        return f"{grade(dF[s]['HI'])} (Si III: {num(dF[s]['SiIII'])}, Δτ/τ {rms((2 * (ta - t0) / (ta + t0))[ta + t0 > 0]):.0%})"
    return grade(dF[s]["HI"])

head = ["rank", "step", "option (fake_spectra's value)", "what fake_spectra does", "SkewerJAX's choice and its reason",
        "rms(ΔF) Lyα", "rms(ΔF) Si III", "Accuracy", "Impact", "Expected cost"]
out = ["| " + " | ".join(head) + " |", "|" + "---|" * len(head)]
for step, (s, option, does, why) in enumerate(ROOT, 1):
    star = " \\*" if s in STARRED else ""
    accuracy, cost = GRADES[s]
    out.append(f"| {rank[s]} | {step}{star} | {option} | {does} | {why} | {num(dF[s]['HI'])} | {num(dF[s]['SiIII'])} | {accuracy} | {impact(s)} | {cost} |")
notes = "; ".join(f"`{s}`, {name}: {num(rms(a['HI']['F'] - b['HI']['F']))} (Lyα), {num(rms(a['SiIII']['F'] - b['SiIII']['F']))} (Si III)" for s, (name, a, b) in own.items())
out += ["", f"\\* takes along what it needs (the table of the switches). In steps 7, 8 and 14, that is in A too, so their numbers and rank include it, and their impact does not. Their own rms(ΔF): {notes}."]
display(Markdown("\n".join(out)))

### What to adopt, what to time

**No-brainers: more accurate at no extra cost.** SkewerJAX's defaults already take all of them.

| step | what changes in the computation | verdict |
|---|---|---|
| 1, 2, 9 | one constant each | yes |
| 3 | both rules evaluate the same fit on every cell and select with `where`; the SFR is read with the other fields | yes, added |
| 4 | $n_{\rm H}$ is computed anyway; no factor | yes |
| 8 | one constant per chord; the default has no chords at all | yes, added |
| 14 | two `log1p` per piece, not per pixel, and `SnapshotFrame` goes through the same line | yes |
| 15 | one `expm1` per Voigt evaluation, the only one here that scales with pieces times pixels: small against ExoJAX, not against Harris | yes with ExoJAX; time it if Harris becomes the default |
| 5 | linear is the cheaper one, but not more accurate everywhere (step 5) | no: equivalent, not better |
| 17 | full profiles are the most accurate and the main cost | no: to time below |

**Choices, by goal.**

| step | SkewerJAX's default | fake_spectra's value |
|---|---|---|
| 11 `velocity_period` | a path through an evolving universe, with ends: nothing re-enters | a periodic box at one epoch, e.g. statistics over the box |
| 12 `a_eff` | the density of a path whose $a$ changes along it, $n_{\rm com}$ frozen at the snapshot | one epoch: the snapshot's density, as stored |
| 13 `hubble_flow` | the physical flow along an evolving path | one epoch: it makes the velocity axis exactly $(aH)_{\rm snap}L$, the period of step 11 (the exact span differs, 0.016 km/s here) |
| 16 `pixel_average` | $\tau$ sampled at the pixel centre, for a model that goes on to its own resolution and rebinning | a per-pixel $\tau$ of narrow cells, when pixels are coarse against $b$; neither is what a spectrograph records |

Steps 11, 12 and 13 belong together: a single-epoch box takes all three, a path through an evolving universe none.

**To time in a later task.** Nothing here has been timed; the counts are from this skewer and the older numbers from a $\mathrm{Ly}\alpha$-only profile of 10 skewers (reply of 2026-11-05).

| what | steps | hypothesis to test |
|---|---|---|
| Voronoi intersections, sphere overlaps, chord quadrature | 6, 7 | $\tau$ is dense in pieces: 17,075, 68,780 and 18,543 here, so 1 : 4 : 1.1 (measured before: 3.9 and 1.1). The Voronoi pieces are then not the expensive ones; what remains is the geometry stage, and the sampling, since 15,847 of the 17,075 pieces come from cutting at the 15,847 dual segments |
| Harris or ExoJAX | 10 | measured before: 13 against 120 ns per evaluation on one core, 2.1 times on 32 threads, $\mathrm{Ly}\alpha$ only. The gain needs every element of a batch at $a \leq 10^{-3}$: Si III is above it for any $T$ below about $10^6$ K, and one `lax.cond` then evaluates both on the whole batch, so with Si III in the bundle Harris may cost more than ExoJAX. Test $\mathrm{Ly}\alpha$ alone, $\mathrm{Ly}\alpha$ + Si III, and a split per line |
| A sparse profile truncation | 17 | today's option evaluates every pixel and then cuts, so it cannot gain. A window per piece at $\epsilon = 10^{-7}$ would keep about 4 % of the evaluations (estimated before); the question is the cost of padded windows under static shapes, against $\mathrm{rms}(\Delta F) \sim 10^{-5}$ |
| $a(y)$: once per path | 12, 13, 14 | their own lines are low, and they share one cost: SkewerJAX solves the ODE once per skewer on its points, then only interpolates per piece. Parallel box-crossing skewers share the same $a(s)$, so one solve could serve them all; or a short series of $\ln a(y)$ about $a_{\rm snap}$; or an effective $a$, which is steps 12 and 13. Measure its share of the run first |
| `expm1` per evaluation | 15 | small against ExoJAX; compare with Harris, and with the series $x + x^2/2$ |